# exp156_test_batch_covariate_context_audit train

exp148 lgb_mean を base に、test batch 内で同時に見える target-free covariate context から high-drift / high-disagreement regime を診断する。LightGBM の新規学習は行わない。

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, load_config
from test_batch_covariate_context_audit import OUTPUT_PREFIX

paths = ExperimentPaths()
config = load_config()
output_dir = paths.artifacts_dir

print('experiment:', config['experiment']['name'])
print('route:', config['experiment']['route'])
print('parent:', config['lineage']['parent'])
print('mode:', config['audit']['mode'])
print('output:', output_dir)
print('gate variants:', [v['name'] for v in config['audit']['gate_variants']])

## 2. Input contract

In [ ]:
input_contract = {
    'exp148_predictions': config['data']['exp148_predictions'],
    'exp073_predictions': config['data']['exp073_predictions'],
    'exp072_feature_cache': config['data']['exp072_feature_cache'],
    'train_dir': config['data']['train_dir'],
    'required_rawtest_compatible_columns': config['audit']['required_rawtest_compatible_columns'],
}
print(json.dumps(input_contract, indent=2))

## 3. Run posthoc audit

In [ ]:
from test_batch_covariate_context_audit import run_train_from_config

summary = run_train_from_config(config, output_dir=output_dir)
print(json.dumps(summary, indent=2)[:6000])

## 4. Metrics and artifacts

In [ ]:
metrics_path = output_dir / f'{OUTPUT_PREFIX}_metrics.csv'
gates_path = output_dir / f'{OUTPUT_PREFIX}_gate_variants.csv'
bucket_path = output_dir / f'{OUTPUT_PREFIX}_bucket_metrics.csv'
common_path = output_dir / f'{OUTPUT_PREFIX}_common_worst_metrics.csv'
parity_path = output_dir / f'{OUTPUT_PREFIX}_rawtest_parity_checklist.csv'

metrics = pd.read_csv(metrics_path)
gates = pd.read_csv(gates_path)
bucket = pd.read_csv(bucket_path)
common = pd.read_csv(common_path)
parity = pd.read_csv(parity_path)

display(metrics.sort_values('rmse').head(20))
display(gates)
display(common.sort_values(['set_name', 'delta_rmse_vs_exp148']).head(30))
display(bucket.sort_values('base_exp148_lgb_mean_rmse', ascending=False).head(30))
display(parity)

print('artifacts:')
for path in sorted(output_dir.glob(f'{OUTPUT_PREFIX}*')):
    print('-', path.name, path.stat().st_size)